# 01 — Exploratory Data Analysis
LinkedIn Job Postings dataset: understanding salary distributions, feature relationships, and class balance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
os.makedirs('../docs/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw/job_postings.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Overview

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
df.describe()

## 3. Create Target Variable — Salary Bands

In [ ]:
# Drop rows missing salary info
df_salary = df.dropna(subset=['max_salary', 'min_salary']).copy()
print(f'Rows with salary info: {len(df_salary)} / {len(df)} ({len(df_salary)/len(df)*100:.1f}%)')

# Normalize to annual (check pay_period column)
print(df_salary['pay_period'].value_counts())

In [ ]:
# Convert hourly to annual if needed
def normalize_salary(row):
    avg = (row['min_salary'] + row['max_salary']) / 2
    if str(row.get('pay_period', '')).upper() == 'HOURLY':
        return avg * 2080  # 40hrs * 52 weeks
    return avg

df_salary['salary_avg'] = df_salary.apply(normalize_salary, axis=1)

# Remove outliers (keep 5th–95th percentile)
low, high = df_salary['salary_avg'].quantile([0.05, 0.95])
df_salary = df_salary[(df_salary['salary_avg'] >= low) & (df_salary['salary_avg'] <= high)]
print(f'After outlier removal: {len(df_salary)} rows')
print(f'Salary range: ${low:,.0f} – ${high:,.0f}')

In [ ]:
# Create salary bands
bins = [0, 50000, 90000, 140000, float('inf')]
labels = ['Low (<50k)', 'Mid (50–90k)', 'High (90–140k)', 'Very High (>140k)']
df_salary['salary_band'] = pd.cut(df_salary['salary_avg'], bins=bins, labels=labels)

print('\nSalary band distribution:')
print(df_salary['salary_band'].value_counts())

## 4. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Salary distribution
axes[0].hist(df_salary['salary_avg'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Salary Distribution (Annual)')
axes[0].set_xlabel('Annual Salary ($)')
axes[0].set_ylabel('Count')

# Class balance
df_salary['salary_band'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Salary Band Class Balance')
axes[1].set_xlabel('Salary Band')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../docs/figures/01_salary_distribution.png', dpi=150)
plt.show()

In [ ]:
# Experience level vs salary band
if 'formatted_experience_level' in df_salary.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    ct = pd.crosstab(df_salary['formatted_experience_level'], df_salary['salary_band'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, colormap='Blues')
    ax.set_title('Salary Band by Experience Level')
    ax.set_xlabel('Experience Level')
    ax.legend(title='Salary Band', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.savefig('../docs/figures/02_experience_vs_salary.png', dpi=150)
    plt.show()

In [ ]:
# Correlation heatmap
num_cols = ['min_salary', 'max_salary', 'salary_avg']
optional_cols = ['applies', 'views']
for c in optional_cols:
    if c in df_salary.columns:
        num_cols.append(c)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df_salary[num_cols].corr(), annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig('../docs/figures/03_correlation_heatmap.png', dpi=150)
plt.show()

## 5. Save Processed Dataset

In [ ]:
cols_to_keep = ['title', 'description', 'salary_avg', 'salary_band',
                'formatted_experience_level', 'formatted_work_type',
                'location', 'company_id']
cols_available = [c for c in cols_to_keep if c in df_salary.columns]
df_clean = df_salary[cols_available].dropna(subset=['salary_band'])

df_clean.to_csv('../data/processed/job_postings_clean.csv', index=False)
print(f'Saved {len(df_clean)} rows to data/processed/job_postings_clean.csv')
df_clean.head()